In [23]:
import os

print("当前工作目录:", os.getcwd())
print("当前文件所在目录:", os.path.dirname(os.path.abspath('__file__')))

# 列出上级目录的内容
parent_dir = os.path.dirname(os.getcwd())
print("上级目录:", parent_dir)
print("上级目录内容:", os.listdir(parent_dir))

当前工作目录: D:\programming\python_script\socio-physical system for shelter
当前文件所在目录: D:\programming\python_script\socio-physical system for shelter
上级目录: D:\programming\python_script
上级目录内容: ['ABM_with_GPU', 'AgentSociety', 'C-LEARNING_VISULIZATION_GRAPH', 'FLAMEGPU2', 'FLAMEGPU2-tutorial-python', 'FLAMEGPU2_python_sugarscape_tutorial', 'geoai_related', 'LITTLE', 'network_select', 'PandemicLLM', 'scrapy', 'SHELTER_FLAMEGPU2', 'shelter_gravity_model', 'sleep_health_lifestyle', 'socio-physical system for shelter', 'test.ipynb', 'test.py', 'testforgit', 'transfer']


In [24]:
import sys
import os

# 切换到你的项目根目录
project_root = r"D:\programming\python_script\socio-physical system for shelter"
os.chdir(project_root)

# 添加 data/output 目录到 Python 路径
sys.path.append('data/output')


## define model

In [25]:
#这个test.py是用来测试pyflame的可视性的

from pyflamegpu import *
import pyflamegpu.codegen
import sys

# Define some useful constants
AGENT_COUNT = 16384
ENV_WIDTH = int(AGENT_COUNT**(1/3))

# Define the FLAME GPU model: 这个可以在后续的可视化窗口改名字
model = pyflamegpu.ModelDescription("First test using default visualization")


## messages setting

In [26]:

# Define a message of type MessageSpatial2D named location
# MessageSpatial2D: Each agent outputs a message at a specific location in 2D space
# agents only read messages located close to a particular search origin（搜素的中心点）.
# 可以获取一定距离内的消息
message = model.newMessageSpatial3D("location")
# Configure the message list
message.setMin(0, 0,0)
message.setMax(ENV_WIDTH, ENV_WIDTH,ENV_WIDTH)
message.setRadius(2)
# Add extra variables to the message
# X Y (Z) are implicit for spatial messages
message.newVariableID("id")



## agent_variables_definition

In [27]:

# Assign the agent some variables (ID is implicit to agents, so we don't define it ourselves)
student_agent = model.newAgent("student_agent")
student_agent.newVariableFloat("x")
student_agent.newVariableFloat("y")
student_agent.newVariableInt("building_id")
student_agent.newVariableInt("point_id")
student_agent.newVariableFloat("z")
student_agent.newVariableFloat("drift", 0)

stairwell_agent = model.newAgent("stairwell_agent")
stairwell_agent.newVariableFloat("x")
stairwell_agent.newVariableFloat("y")
stairwell_agent.newVariableInt("stairwell_id")
stairwell_agent.newVariableFloat("z")



## environment setting

In [28]:

# Define environment properties
env = model.Environment()
env.newPropertyUInt("AGENT_COUNT", AGENT_COUNT)
env.newPropertyFloat("ENV_WIDTH", ENV_WIDTH)
env.newPropertyFloat("repulse", 0.05)


## agent function

In [ ]:
@pyflamegpu.agent_function
def set_target_stairwell(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )

In [29]:

@pyflamegpu.agent_function
def output_message(message_in: pyflamegpu.MessageNone, message_out: pyflamegpu.MessageSpatial3D):
    message_out.setVariableUInt("id", pyflamegpu.getID())
    message_out.setLocation(
        pyflamegpu.getVariableFloat("x"),
        pyflamegpu.getVariableFloat("y"),
        pyflamegpu.getVariableFloat("z")
        )
    return pyflamegpu.ALIVE

@pyflamegpu.agent_function
def input_message(message_in: pyflamegpu.MessageSpatial3D, message_out: pyflamegpu.MessageNone):
    ID = pyflamegpu.getID()
    REPULSE_FACTOR = pyflamegpu.environment.getPropertyFloat("repulse")
    RADIUS = message_in.radius()
    fx = 0.0
    fy = 0.0
    fz = 0.0
    x1 = pyflamegpu.getVariableFloat("x")
    y1 = pyflamegpu.getVariableFloat("y")
    z1 = pyflamegpu.getVariableFloat("z")
    count = 0 
    for message in message_in(x1, y1, z1):
        if message.getVariableUInt("id") != ID :
            x2 = message.getVariableFloat("x")
            y2 = message.getVariableFloat("y")
            z2 = message.getVariableFloat("z")
            x21 = x2 - x1
            y21 = y2 - y1
            z21 = z2 - y2
            separation = math.sqrtf(x21*x21 + y21*y21 + z21*z21)
            if separation < RADIUS and separation > 0 :
                k = math.sinf((separation / RADIUS)*3.141*-2)*REPULSE_FACTOR
                # Normalise without recalculating separation
                x21 /= separation
                y21 /= separation
                z21 /= separation
                fx += k * x21
                fy += k * y21
                fz += k* z21
                count += 1
    fx /= count if count > 0 else 1
    fy /= count if count > 0 else 1
    fz /= count if count > 0 else 1
    pyflamegpu.setVariableFloat("x", x1 + fx)
    pyflamegpu.setVariableFloat("y", y1 + fy)
    pyflamegpu.setVariableFloat("z", z1 + fz)

    pyflamegpu.setVariableFloat("drift", math.sqrtf(fx*fx + fy*fy + fz*fz))
    return pyflamegpu.ALIVE


## function write in

In [30]:

# translate the agent functions from Python to C++
output_func_translated = pyflamegpu.codegen.translate(output_message)
input_func_translated = pyflamegpu.codegen.translate(input_message)
# Setup the two agent functions
out_fn = student_agent.newRTCFunction("output_message", output_func_translated)
out_fn.setMessageOutput("location")
in_fn = student_agent.newRTCFunction("input_message", input_func_translated)
in_fn.setMessageInput("location")

# Message input depends on output
in_fn.dependsOn(out_fn)

# 添加学生代理类型
# 基于data/output/flamegpu_init_code.py的学生代理初始化


# Dependency specification
# Output is the root of our graph
model.addExecutionRoot(out_fn)
model.generateLayers()



## simulation creation

In [31]:

# Create and init the simulation
cuda_model = pyflamegpu.CUDASimulation(model)



## initialization

In [32]:


from flamegpu_init_code import initialize_student_agent_population

# 初始化学生代理种群
initialize_student_agent_population(model, cuda_model)

# Specify the desired StepLoggingConfig
step_log_cfg = pyflamegpu.StepLoggingConfig(model)
# Log every step
step_log_cfg.setFrequency(1) 
# Include the mean of the "point" agent population's variable 'drift'
step_log_cfg.agent("student_agent").logMeanFloat("drift")
step_log_cfg.agent("student_agent").logMeanFloat("x")
step_log_cfg.agent("student_agent").logMeanFloat("y")


cuda_model.initialise(sys.argv)


# Attach the logging config
cuda_model.setStepLog(step_log_cfg)


初始化 6490 个学生代理个体
学生代理种群初始化完成
初始化 28 个楼梯间代理
楼梯间代理种群初始化完成


## visualization

In [33]:

WIDTH=500

# Only run this block if pyflamegpu was built with visualisation support
if pyflamegpu.VISUALISATION:
    # Create visualisation
    m_vis = cuda_model.getVisualisation()
    # Set the initial camera location and speed
    INIT_CAM = WIDTH / 2
    m_vis.setInitialCameraTarget(270, 205, 0)
    m_vis.setInitialCameraLocation(240, 100, 100)
    m_vis.setCameraSpeed(0.01)
    m_vis.setSimulationSpeed(25)
    # Add "point" agents to the visualisation

    
    # Add "student_agent" agents to the visualisation
    student_agt = m_vis.addAgent("student_agent")
    student_agt.setModel(pyflamegpu.ICOSPHERE);
    student_agt.setModelScale(1/1.0);
    # Mark the environment bounds.

    stairwell_agt = m_vis.addAgent("stairwell_agent")
    stairwell_agt.setModel(pyflamegpu.ICOSPHERE);
    stairwell_agt.setModelScale(1/0.5);
    stairwell_agt.setColor(pyflamegpu.RED);
    
    pen = m_vis.newPolylineSketch(1, 1, 1, 0.2)
    pen.addVertex(275, 637, 0) # 起始点
    pen.addVertex(69, 510, 0)
    pen.addVertex(0, 301, 0)
    pen.addVertex(1, 167, 0)
    pen.addVertex(29, 142, 0)
    pen.addVertex(57, 98, 0)
    pen.addVertex(118, 67, 0)
    pen.addVertex(109, 24, 0)
    pen.addVertex(287, 0, 0)
    pen.addVertex(286, 45, 0)
    pen.addVertex(405, 154, 0)
    pen.addVertex(435, 131, 0)
    pen.addVertex(436, 72, 0)
    pen.addVertex(467, 41, 0)
    pen.addVertex(501, 35, 0)
    pen.addVertex(543, 47, 0)
    pen.addVertex(275, 637, 0) # 闭合点 
    # Open the visualiser window 
    m_vis.activate()

# Run the simulation
for i in range(100):
        cuda_model.step()



if pyflamegpu.VISUALISATION:
    # Keep the visualisation window active after the simulation has completed
    m_vis.join()

## data collection

In [37]:
import numpy as np

out_pop = pyflamegpu.AgentVector(model.Agent("student_agent"))
cuda_model.getPopulationData(out_pop)

# 创建结构化数组
dtype = [('building_id', 'i4'),('x','f4'),('y','f4'),('z', 'f4')]
agent_array = np.array(
    [(agent.getVariableInt("building_id"), agent.getVariableFloat("x"),agent.getVariableFloat("y"),agent.getVariableFloat("z")) 
     for agent in out_pop],
    dtype=dtype
)

agent_array

array([( 0, 272.11014, 269.37363,  6.), ( 1, 338.47827, 303.36465,  3.),
       ( 2, 363.70874, 301.3737 ,  9.), ...,
       (11, 188.49626, 428.31555, 21.), (11, 171.74149, 391.04172,  6.),
       (13, 306.83536, 457.45065, 15.)],
      dtype=[('building_id', '<i4'), ('x', '<f4'), ('y', '<f4'), ('z', '<f4')])

In [35]:
out_pop = pyflamegpu.AgentVector(model.Agent("student_agent"))
cuda_model.getPopulationData(out_pop)
for agent in out_pop:
    print(" building_id %f"%(agent.getVariableInt("building_id")))
    print(" z %f"%(agent.getVariableFloat("z")))

 building_id 0.000000
 z 6.000000
 building_id 1.000000
 z 3.000000
 building_id 2.000000
 z 9.000000
 building_id 2.000000
 z 9.000000
 building_id 3.000000
 z 15.000000
 building_id 3.000000
 z 18.000000
 building_id 3.000000
 z 18.000000
 building_id 4.000000
 z 18.000000
 building_id 7.000000
 z 18.000000
 building_id 7.000000
 z 18.000000
 building_id 10.000000
 z 3.000000
 building_id 10.000000
 z 39.000000
 building_id 10.000000
 z 6.000000
 building_id 10.000000
 z 21.000000
 building_id 10.000000
 z 15.000000
 building_id 11.000000
 z 15.000000
 building_id 13.000000
 z 12.000000
 building_id 2.000000
 z 18.000000
 building_id 2.000000
 z 12.000000
 building_id 3.000000
 z 21.000000
 building_id 3.000000
 z 6.000000
 building_id 4.000000
 z 12.000000
 building_id 4.000000
 z 3.000000
 building_id 4.000000
 z 18.000000
 building_id 4.000000
 z 21.000000
 building_id 6.000000
 z 3.000000
 building_id 6.000000
 z 18.000000
 building_id 6.000000
 z 21.000000
 building_id 6.000000
